In [6]:
# Brochure generation with multi-shot prompting

In [18]:
import os
import requests
import json

from typing import List
from dotenv import load_dotenv
from openai import OpenAI
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display

In [19]:
# Constants and environment

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'


api_key = os.getenv("OPENAI_API_KEY")
load_dotenv(override=True)

if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    print("API KEY looks good so far")
else:
    print("API key looks invalid. Please see the troubleshooting documentation")

open_ai = OpenAI()
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


API KEY looks good so far


In [20]:
# Website class

headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [21]:
ed = Website("https://edwarddonner.com")
ed.links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/',
 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/',
 'https://edwarddonner.com/2025/04/21/the-complete-agentic-ai-engineering-course/',
 'https://edwarddonner.com/2025/04/21/the-complete-agentic-ai-engineering-course/',
 'https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/',
 'https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-wit

In [36]:
# Link Handling

link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""
link_system_prompt += "So if you were provided a list of links such as this:\n"
link_system_prompt += """['https://webpage.com/',
'https://webpage.com/about',
'https://www.linkdin.com/in/webpage',
'https://www.twitter.com/webpage',
'https://webpage.com/mission',
'https://webpage.com/careers',
'https://news.com/new-business-ventures-2025',
'https://hello@domainname.com',
'https://webpage.com/posts'] \n"""
link_system_prompt += "You would reply like this:\n"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://webpage.com/about"},
        {"type": "mission page", "url": "https://webpage.com/mission"},
        {"type": "careers page", "url": "https://webpage.com/careers"},
        {"type": "twitter page", "url": "https://twitter.com/webpage"},
        {"type": "linkdin page", "url": "https://linkdin.com/in/webpage"}
    ]
}
"""

def get_link_user_message(site: Website):
    message = f"Here is the list of links on the website of {site.url} - "
    message += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    message += "Links (some might be relative links):\n"
    message += "\n".join(site.links)
    return message

def filter_links(site: Website):
    response = open_ai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_link_user_message(site)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [37]:
test = Website("https://edwarddonner.com")
filter_links(test)

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'linkedin page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter page', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [53]:
# Brochure handling

main_system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

def get_website_details(site: Website):
    results = 'Landing page:\n'
    results += site.get_contents()
    links = filter_links(site)
    for link in links["links"]:
        results += f"\n\n{link['type']}\n"
        results += Website(link["url"]).get_contents()
    return results

def create_brochure(site: Website):
    stream = open_ai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": main_system_prompt},
            {"role": "user", "content": get_website_details(site)}
        ],
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)
    return response


brochure = create_brochure(ed)

# Nebula.io: Empowering Talent through AI

### About Us
At **Nebula.io**, we are revolutionizing the recruitment landscape by applying cutting-edge **Generative AI** and machine learning technology. Our **patented matching model** allows recruiters to source, understand, engage, and manage talent with unprecedented accuracy and efficiency - without relying on keywords. This innovative approach is grounded in the belief that everyone has the right to discover their true potential and pursue their **Ikigai**—the intersection of what they love, what they are good at, what the world needs, and what they can be rewarded for.

Our mission is poignant: to elevate human prosperity by finding people their dream jobs in a world where 77% of individuals feel uninspired or disengaged at work. Together, we’re not just filling roles; we’re transforming lives.

### Company Culture
Our company culture at Nebula.io is fueled by **innovation, collaboration, and a commitment to making a positive impact**. We foster an environment where creative ideas thrive, encouraging team members to experiment and contribute to our groundbreaking products. We celebrate each individual’s unique path and believe in the power of diverse perspectives to drive our success. 

We are passionate about what we do and approach challenges with a sense of curiosity and resilience. We prioritize a work-life balance that allows our team members to cultivate their own passions, which includes DJing, electronic music production, and engaging with broader tech communities like **Hacker News**.

### Our Customers
Our primary customers include recruitment firms and organizations looking for a more effective way to engage with potential hires. By leveraging our proprietary LLMs (Large Language Models) tailored specifically for the talent acquisition domain, we help these companies not just see resumes, but understand the essence of candidates, matching them to the roles where they will make the most impact.

### Careers
Are you ready to join an innovative team that is redefining the future of work? At Nebula.io, we are always looking for visionary thinkers, aspiring engineers, and AI enthusiasts who are passionate about making a difference in the world of recruitment. We offer various career paths ranging from engineering and data science to customer success and product development.

**Join Us**: If you are curious about applying AI to solve real-world problems, believe in the power of technology for personal fulfillment, and want to grow alongside inspiring like-minded professionals, explore your opportunities with us.

### Connect with Us
We invite you to connect with us, whether you’re a prospective customer, investor, or future team member. Engage with our community and explore how we can work together to transform talent acquisition. 

For more information and to try our platform for free, visit us at [Nebula.io](http://www.nebula.io). 

### Get in Touch
For inquiries or to schedule a virtual coffee, reach out to Ed Donner at ed[at]edwarddonner[dot]com.

---

**Nebula.io**: Where the future of talent meets technology. Let's build a better workforce together!

In [56]:
# Translation handling

translate_system_message = "You are provided with a document in markdown format. \
You are able to translate this document into a specified language, \
retaining any markdown formatting."


def get_translate_user_message(document: str, lang: str):
    message = f"Translate this document into {lang}:\n"
    message+= document
    return message

def translate_document(document: str, to_lang: str):
    stream = open_ai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": translate_system_message},
            {"role": "user", "content": get_translate_user_message(document, to_lang)}
        ],
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
translate_document(brochure, "German")